# 01 · ETL, validação e engenharia de atributos

**Trabalho N3 – Análise Preditiva (MLOps)** · Etapas 1, 2 e entrada de dados do fluxo único.

## Etapa 1 – Problema, dados e avaliação

| Item | Definição |
|---|---|
| **Dataset** | Bike Sharing Dataset – `hour.csv` (Capital Bikeshare, Washington D.C., 2011–2012) |
| **Fonte / licença** | UCI Machine Learning Repository (id 275) – licença **CC BY 4.0** |
| **Tipo de problema** | **Regressão** – prever a demanda horária de aluguéis |
| **Unidade de observação** | 1 hora de operação do sistema, chave composta `["dteday", "hr"]` |
| **Variável-alvo** | `cnt` – total de bicicletas alugadas na hora |
| **Atributos** | 12 originais (calendário + clima normalizados) + 8 criados no ETL = 20 candidatos |
| **Métrica principal** | **MAE** (direção `minimize`) – erro médio em *bicicletas/hora*, interpretável pela operação e menos sensível aos picos extremos que o RMSE. RMSE e R² são secundárias. |

**Prevenção de vazamento.** `instant` (ID sequencial) é removido; `casual` e `registered` são removidos porque `casual + registered = cnt` (decompõem o próprio alvo). `dteday` só é usado para ordenação, corte temporal e deduplicação – nunca entra no modelo.

**Protocolo de particionamento (definido antes do treino).**

| Conjunto | Período | Uso |
|---|---|---|
| `lote_1.csv` (v1) | 2011-01-01 → 2012-06-30 | dados do ciclo inicial |
| `lote_2.csv` (v2) | 2012-07-01 → 2012-10-31 | lote novo, incorporado no ciclo seguinte (sem duplicar a chave) |
| validação interna | últimos 20% cronológicos do ciclo ativo | escolher vencedor e comparar com o vigente |
| `teste_final.csv` | 2012-11-01 → 2012-12-31 | **reservado**: só no notebook 05 |

**Chegada de novos registros.** Cada execução agendada incorpora o próximo lote pendente listado em `config.yaml`; os lotes já processados ficam em `models/estado_pipeline.json`.

**Limitações.** Dados de uma única cidade e de apenas 2 anos (tendência de crescimento captada só por `yr`); variáveis climáticas normalizadas (sem unidades físicas); existem horas sem registro (horas com zero aluguel não foram gravadas); o teste final cobre só novembro/dezembro (inverno).

In [1]:
# Setup: garante caminhos consistentes (nbconvert executa com cwd = notebooks/)
import os, sys, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import mlflow

from src import utils
cfg = utils.carregar_config()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: /home/runner/work/bsd/bsd


## 1. Definição do ciclo (`cycle_id` e `data_version`)

In [2]:
from src import ingestao, preprocessamento as pp

estado = ingestao.carregar_estado()
ciclo = ingestao.definir_ciclo(cfg, estado)
print(json.dumps(ciclo, indent=2, ensure_ascii=False))
DIR_EV = utils.dir_evidencias(ciclo["cycle_id"])

{
  "cycle_id": "ciclo_1",
  "numero_execucao": 1,
  "data_version": "v1",
  "lotes_ciclo": [
    "lote_1.csv"
  ],
  "lote_novo": "lote_1.csv",
  "lotes_anteriores": [],
  "hash_lotes": {
    "lote_1.csv": "bf5de8a10b861d44db9c8084b6ead90ec29037822e9e257e5e17b7e8848f8def"
  },
  "iniciado_em": "2026-11-11T11:28:35+00:00",
  "gatilho": "workflow_dispatch",
  "github_run_id": "43012084623"
}


## 2. Entrada e incorporação dos lotes (sem duplicação pela chave `dteday`+`hr`)

In [3]:
bruto, rel_incorporacao = ingestao.incorporar_lotes(cfg, ciclo["lotes_ciclo"])
print(json.dumps(rel_incorporacao, indent=2))
display(bruto.head())
display(bruto.groupby("lote_origem")["dteday"].agg(["min", "max", "count"]))

{
  "linhas_por_lote": {
    "lote_1.csv": 13003
  },
  "linhas_concatenadas": 13003,
  "duplicatas_chave_removidas": 0,
  "linhas_consolidadas": 13003
}


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,lote_origem
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,lote_1.csv
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,lote_1.csv
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,lote_1.csv
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,lote_1.csv
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,lote_1.csv


,min,max,count
lote_origem,,,
lote_1.csv,2011-01-01,2012-06-30,13003


## 3. Diagnóstico **antes** do tratamento

In [4]:
diag_antes = pp.diagnosticar(bruto, cfg)
print(json.dumps(diag_antes, indent=2, ensure_ascii=False))
display(bruto.describe().T)

{
  "linhas": 13003,
  "periodo": [
    "2011-01-01",
    "2012-06-30"
  ],
  "ausentes": {},
  "total_ausentes": 0,
  "duplicatas_chave": 0,
  "duplicatas_linha": 0,
  "fora_do_dominio": {},
  "umidade_zero": 22,
  "inconsistencia_casual_registered": 0,
  "inconsistencia_workingday": 0,
  "outliers_iqr": {
    "cnt": 406,
    "windspeed": 87,
    "hum": 0,
    "temp": 0
  },
  "horas_esperadas": 13128,
  "horas_ausentes_no_periodo": 125
}


,count,mean,std,min,25%,50%,75%,max
instant,13003.0,6502.000000,3753.787110,1.00,3251.5000,6502.0000,9752.5000,13003.0000
season,13003.0,2.212874,1.052636,1.00,1.0000,2.0000,3.0000,4.0000
yr,13003.0,0.335153,0.472062,0.00,0.0000,0.0000,1.0000,1.0000
mnth,13003.0,5.544720,3.300547,1.00,3.0000,5.0000,8.0000,12.0000
hr,13003.0,11.555333,6.911492,0.00,6.0000,12.0000,18.0000,23.0000
holiday,13003.0,0.027455,0.163412,0.00,0.0000,0.0000,0.0000,1.0000
weekday,13003.0,3.009536,2.004472,0.00,1.0000,3.0000,5.0000,6.0000
workingday,13003.0,0.684765,0.464627,0.00,0.0000,1.0000,1.0000,1.0000
weathersit,13003.0,1.426286,0.647792,1.00,1.0000,1.0000,2.0000,4.0000
temp,13003.0,0.481620,0.191224,0.02,0.3200,0.4800,0.6400,0.9800


## 4. Limpeza e engenharia de atributos

Tratamentos (implementados em `src/preprocessamento.py`):

* remoção das colunas de vazamento e de duplicatas pela chave;
* valores fora do domínio físico e `hum == 0` (umidade relativa nula é leitura impossível) → `NaN`; a **imputação pela mediana é ajustada apenas no treino**, dentro do `Pipeline`;
* outliers de `cnt` e `windspeed` **mantidos** – picos de demanda e rajadas são fenômenos reais;
* 8 novos atributos: `hr_sin`, `hr_cos`, `mnth_sin`, `mnth_cos` (codificação cíclica), `is_rush_hour`, `is_weekend`, `temp_diff` (`atemp - temp`) e `temp_hum_interaction` (`temp * hum`). São transformações linha a linha (sem estatísticas globais), portanto não há vazamento.

In [5]:
dados, acoes = pp.preparar(bruto, cfg)
print(json.dumps(acoes, indent=2, ensure_ascii=False))
diag_depois = pp.diagnosticar(dados, cfg)

resumo = pd.DataFrame({
    "antes": {
        "linhas": diag_antes["linhas"],
        "colunas": bruto.shape[1],
        "ausentes": diag_antes["total_ausentes"],
        "duplicatas_chave": diag_antes["duplicatas_chave"],
        "valores_fora_dominio": sum(diag_antes["fora_do_dominio"].values()),
        "umidade_zero": diag_antes["umidade_zero"],
        "colunas_vazamento": sum(c in bruto for c in cfg["dados"]["colunas_vazamento"]),
    },
    "depois": {
        "linhas": diag_depois["linhas"],
        "colunas": dados.shape[1],
        "ausentes": diag_depois["total_ausentes"],
        "duplicatas_chave": diag_depois["duplicatas_chave"],
        "valores_fora_dominio": sum(diag_depois["fora_do_dominio"].values()),
        "umidade_zero": diag_depois["umidade_zero"],
        "colunas_vazamento": sum(c in dados for c in cfg["dados"]["colunas_vazamento"]),
    },
})
resumo.to_csv(DIR_EV / "etl_antes_depois.csv")
display(resumo)
print("Obs.: os 'ausentes' depois correspondem aos valores marcados como NaN (ex.: hum == 0),"
      " que serão imputados dentro do Pipeline com estatísticas apenas do treino.")

{
  "colunas_removidas_vazamento": [
    "instant",
    "casual",
    "registered"
  ],
  "duplicatas_removidas": 0,
  "fora_do_dominio_para_nan": {},
  "umidade_zero_para_nan": 22,
  "linhas_sem_alvo_removidas": 0
}


,antes,depois
linhas,13003,13003
colunas,18,23
ausentes,0,44
duplicatas_chave,0,0
valores_fora_dominio,0,0
umidade_zero,22,0
colunas_vazamento,3,0


Obs.: os 'ausentes' depois correspondem aos valores marcados como NaN (ex.: hum == 0), que serão imputados dentro do Pipeline com estatísticas apenas do treino.


In [6]:
# Validações automáticas: falhas interrompem o pipeline (e impedem promoção indevida)
features = pp.lista_features(cfg)
assert len(features) == 20, "Devem existir 20 features candidatas"
assert not set(cfg["dados"]["colunas_vazamento"]) & set(dados.columns), "Coluna de vazamento presente"
assert dados.duplicated(subset=cfg["dados"]["chave"]).sum() == 0, "Chave duplicada"
assert dados[cfg["dados"]["alvo"]].notna().all() and (dados[cfg["dados"]["alvo"]] >= 0).all()
assert dados["dteday"].is_monotonic_increasing, "Dados fora de ordem cronológica"
assert set(features).issubset(dados.columns)
print("Validações OK -", len(dados), "linhas,", len(features), "features candidatas")
display(dados[features + ["cnt"]].describe().T)

Validações OK - 13003 linhas, 20 features candidatas


,count,mean,std,min,25%,50%,75%,max
season,13003.0,2.212874,1.052636,1.0000,1.000000,2.000000e+00,3.000000,4.0000
yr,13003.0,0.335153,0.472062,0.0000,0.000000,0.000000e+00,1.000000,1.0000
mnth,13003.0,5.544720,3.300547,1.0000,3.000000,5.000000e+00,8.000000,12.0000
hr,13003.0,11.555333,6.911492,0.0000,6.000000,1.200000e+01,18.000000,23.0000
holiday,13003.0,0.027455,0.163412,0.0000,0.000000,0.000000e+00,0.000000,1.0000
weekday,13003.0,3.009536,2.004472,0.0000,1.000000,3.000000e+00,5.000000,6.0000
workingday,13003.0,0.684765,0.464627,0.0000,0.000000,1.000000e+00,1.000000,1.0000
weathersit,13003.0,1.426286,0.647792,1.0000,1.000000,1.000000e+00,2.000000,4.0000
temp,13003.0,0.481620,0.191224,0.0200,0.320000,4.800000e-01,0.640000,0.9800
atemp,13003.0,0.462656,0.171211,0.0000,0.318200,4.697000e-01,0.606100,1.0000


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
diario = dados.groupby(["dteday", "lote_origem"])["cnt"].sum().reset_index()
for lote, g in diario.groupby("lote_origem"):
    axes[0].plot(g["dteday"], g["cnt"], lw=0.7, label=lote)
axes[0].set_title("Demanda diária por lote"); axes[0].legend(fontsize=8)
dados.groupby(["hr", "workingday"])["cnt"].mean().unstack().plot(ax=axes[1])
axes[1].set_title("Demanda média por hora (workingday)")
axes[2].hist(dados["cnt"], bins=50); axes[2].set_title("Distribuição do alvo cnt")
fig.tight_layout(); fig.savefig(DIR_EV / "etl_visao_geral.png", dpi=110); plt.show()

## 5. Monitoramento de *data drift* do lote novo (Critério 10 – inovação)

**Problema:** os lotes chegam ao longo do tempo (sazonalidade, crescimento do sistema). Um modelo treinado em dados antigos pode degradar silenciosamente.
**Solução:** ao incorporar um lote novo, comparar sua distribuição com o histórico (teste **Kolmogorov-Smirnov** + **PSI**) nas variáveis contínuas e nas **previsões do modelo vigente**. Alerta = KS significativo (p < 0,05) **e** PSI > 0,2. O relatório vai para `evidencias/`, para o MLflow e para o resumo da execução no GitHub Actions.

In [8]:
from src import monitoramento, promocao

relatorio_drift = None
if ciclo["lote_novo"] and ciclo["lotes_anteriores"]:
    ref = dados[dados["lote_origem"].isin(ciclo["lotes_anteriores"])]
    novo = dados[dados["lote_origem"] == ciclo["lote_novo"]]
    pred_ref = pred_novo = None
    manifesto = promocao.carregar_manifesto()
    if manifesto:
        vigente = promocao.carregar_modelo_vigente(manifesto)
        pred_ref, pred_novo = vigente.predict(ref), vigente.predict(novo)
    relatorio_drift = monitoramento.relatorio_drift(ref, novo, cfg, pred_ref, pred_novo)
    relatorio_drift["lote_novo"] = ciclo["lote_novo"]
    relatorio_drift["modelo_vigente_usado"] = manifesto["modelo_id"] if manifesto else None
    utils.salvar_json(relatorio_drift, DIR_EV / "drift_relatorio.json")
    tab = monitoramento.tabela_drift(relatorio_drift)
    tab.to_csv(DIR_EV / "drift_tabela.csv")
    display(tab)
    monitoramento.grafico_drift(ref, novo, cfg["monitoramento"]["drift"]["variaveis"],
                                DIR_EV / "drift_distribuicoes.png", pred_ref, pred_novo)
    display(Image(DIR_EV / "drift_distribuicoes.png"))
    print("ALERTA DE DRIFT:" if relatorio_drift["alerta"] else "Sem drift relevante.",
          relatorio_drift["variaveis_com_drift"])
else:
    print("Sem lote novo com histórico anterior para comparar (ciclo inicial ou retreino sem dados novos).")

Sem lote novo com histórico anterior para comparar (ciclo inicial ou retreino sem dados novos).


## 6. Persistência do dataset do ciclo e registro no MLflow

In [9]:
utils.DIR_PROCESSED.mkdir(parents=True, exist_ok=True)
dados.to_parquet(utils.DIR_PROCESSED / "dataset_ciclo.parquet", index=False)

exp_id = utils.configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']}_01_etl", tags=utils.tags_ciclo(ciclo, "01_etl")) as run:
    mlflow.log_params({
        "cycle_id": ciclo["cycle_id"],
        "data_version": ciclo["data_version"],
        "lotes_ciclo": ",".join(ciclo["lotes_ciclo"]),
        "lote_novo": ciclo["lote_novo"] or "nenhum",
        "colunas_removidas": ",".join(acoes["colunas_removidas_vazamento"]),
        "n_features_candidatas": len(features),
    })
    mlflow.log_metrics({
        "linhas_consolidadas": len(dados),
        "duplicatas_chave_removidas": rel_incorporacao["duplicatas_chave_removidas"],
        "ausentes_antes": diag_antes["total_ausentes"],
        "umidade_zero_corrigida": acoes["umidade_zero_para_nan"],
    })
    mlflow.log_dict({"antes": diag_antes, "depois": diag_depois, "acoes": acoes,
                     "incorporacao": rel_incorporacao, "hash_lotes": ciclo["hash_lotes"]},
                    "etl/diagnostico.json")
    mlflow.log_artifact(str(DIR_EV / "etl_visao_geral.png"), "etl")
    if relatorio_drift:
        mlflow.log_metrics({f"drift_psi_{v}": r["psi"] for v, r in relatorio_drift["variaveis"].items()})
        mlflow.log_metric("drift_psi_maximo", relatorio_drift["psi_maximo"])
        mlflow.log_metric("drift_alerta", int(relatorio_drift["alerta"]))
        if relatorio_drift["previsoes_vigente"]:
            mlflow.log_metric("drift_psi_previsoes", relatorio_drift["previsoes_vigente"]["psi"])
        mlflow.set_tag("drift_alerta", str(relatorio_drift["alerta"]))
        mlflow.log_artifact(str(DIR_EV / "drift_relatorio.json"), "drift")
        mlflow.log_artifact(str(DIR_EV / "drift_distribuicoes.png"), "drift")

ciclo["run_id_etl"] = run.info.run_id
ciclo["drift_alerta"] = relatorio_drift["alerta"] if relatorio_drift else None
utils.salvar_json({"antes": diag_antes, "depois": diag_depois, "acoes": acoes,
                   "incorporacao": rel_incorporacao}, DIR_EV / "etl_diagnostico.json")
ingestao.registrar_ciclo(ciclo)
print("Run ETL:", run.info.run_id, "| ciclo registrado em", utils.relativo(utils.CAMINHO_CICLO))

2026/11/11 11:28:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/11 11:28:36 INFO mlflow.store.db.utils: Updating database tables


Run ETL: 1899877c09d448d9ad326e9ed97c4939 | ciclo registrado em data/processed/ciclo_atual.json
